# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/EXP91/cellpose/output/plots"
root_dir = "/ceph.groups/mshahbazi.grp/rsakata/EXP91/cellpose/output/1_cellpose"
sample_sheet_csv = "/ceph.groups/mshahbazi.grp/rsakata/EXP91/sample_sheet.csv"
#cyto_csv ="/ceph.groups/mshahbazi.grp/rsakata/EXP77/cellpose/output/3_cytomask_intensities/setA_cyto_intensity.csv"

filter_dapi = FALSE

In [ ]:
EXP = "EXP91"
#EXP_SUB = "T2"

In [ ]:
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )


In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F62", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP_group = c("pos"= "#86AB30","neg"="#8d8d8dff")

col_RFP_group = c("pos"= "#EB5951","neg"="#8d8d8dff")


col_GATA3 = "#489C9C"
col_NANOG = "#EA9542"
col_neg    = "#8d8d8dff"

col_GFP = "#86AB30"
col_RFP = "#EB5951"

## 1. Extract summary files

In [ ]:
if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

In [ ]:
# Find matching CSVs in all subfolders
csv_paths <- list.files(
  path = root_dir,
  pattern = "\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

if (length(csv_paths) == 0) stop("No matching files found.")

# Read and bind
combined_df <- purrr::map_dfr(csv_paths, ~ readr::read_csv(.x, show_col_types = FALSE))


#rename colums and add new column for sample number
combined_df <- combined_df %>%
  rename(image = sample) %>%
  mutate(sample = str_replace(image, "_.*$", ""))

combined_df$sample <- as.character(combined_df$sample)


In [ ]:
head(combined_df)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv(sample_sheet_csv, show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- combined_df %>%
  left_join(sample_sheet, by = "sample")   # keeps all rows from df1

In [ ]:
sample_sheet

In [ ]:
head(merged_df)

In [ ]:
merged_df <- merged_df %>% filter(!image %in% c(
  "2_3","3_1-2","3_6-2","3_9-2","3_9-3","5_1-2","5_4-1",
  "5_5-2","5_9-1","5_11","6_3-2","6_6-1","6_7","6_9",
  "6_10-1","6_11-1","6_11-2","6_12-1","6_13"
))

In [ ]:
tbl <- merged_df %>%
  group_by(sample, sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") %>%
  arrange(sample)  # optional

tbl

In [ ]:
merged_df$exp = EXP
#merged_df$exp_sub = EXP_SUB

## 2. Preprocess

In [ ]:
colnames(merged_df)

In [ ]:
head(merged_df)

### Normalise intensities

In [ ]:
title = "GFP_dapi"

w <- 3.5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_GFP, color = condition)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " Mean_dapi", y = " Mean_GFP", title = "") +
  #geom_abline(intercept = 0, slope = GFP_thresh , linetype = "dashed")+
  settheme+
  scale_color_manual(values=col_condition)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
title = "GATA3_dapi"

w <- 3.5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_GATA3, color = condition)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " Mean_dapi", y = " Mean_GATA3", title = "") +
  settheme+
  scale_color_manual(values=col_condition)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
#Compute the normalised intensities
#merged_df$GFP_norm = merged_df$Mean_GFP/merged_df$Mean_dapi
merged_df$GFP_norm = merged_df$Mean_GFP/merged_df$Mean_dapi
merged_df$GATA3_norm = merged_df$Mean_GATA3/merged_df$Mean_dapi
merged_df$NANOG_norm = merged_df$Mean_NANOG/merged_df$Mean_dapi
merged_df$mcherry_norm = merged_df$Mean_mcherry/merged_df$Mean_dapi


## Plot 

### C) Intensity threshold jitter

In [ ]:
library(ggplot2)
library(rlang)

plot_jitter <- function(
  data,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir,
  title = "",
  w = 5, h = 3,
  palette = NULL,
  hline_at = NULL,                 # <— add a dotted horizontal line at this y
  hline_color = "gray30",
  hline_size = 0.5,
  hline_lty =  "dashed"
) {
  x <- rlang::enquo(x); y <- rlang::enquo(y); color <- rlang::enquo(color)

  p <- ggplot(data, aes(x = fct_rev(!!x), y = !!y, color = !!color)) +
    geom_jitter(width = 0.2, size = 0.7, alpha = 0.6, na.rm = TRUE) +
    labs(x = "", y = "normalised intensity", title = title) +
    settheme +
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
    coord_flip()

  if (!is.null(palette)) p <- p + scale_color_manual(values = palette)
  if (!is.null(hline_at)) p <- p + geom_hline(yintercept = hline_at, linetype = hline_lty,
                                              linewidth = hline_size, color = hline_color)

  ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
  p
}


### D) Intensity threshold_hist

In [ ]:
plot_hist <- function(
  data,
  x = GATA3_norm,
  facet = sample_name,
  out_dir,
  title = "GATA3_norm_hist",
  w = 5, h = 5,
  bins = 30,
  binwidth = NULL,
  fill = "#6CD1D4",
  outline = "gray10",
  linewidth = 0.2,
  free_y = TRUE,
  vline_at = NULL,              # <— vertical line at this x
  vline_color = "gray30",
  vline_size = 0.5,
  vline_lty = "dashed"
) {
  x     <- rlang::enquo(x)
  facet <- rlang::enquo(facet)

  p <- ggplot2::ggplot(data, ggplot2::aes(x = !!x)) +
    (if (!is.null(binwidth))
       ggplot2::geom_histogram(binwidth = binwidth, boundary = 0, closed = "left",
                                na.rm = TRUE, fill = fill, color = outline, linewidth = linewidth)
     else
       ggplot2::geom_histogram(bins = bins, na.rm = TRUE,
                                fill = fill, color = outline, linewidth = linewidth)) +
    ggplot2::labs(x = "normalised intensity", y = "Count", title = title) +
    settheme +
    ggplot2::scale_y_continuous(limits = c(0, NA), expand = c(0, 0),
      breaks = scales::pretty_breaks(n = 2)   # <— fewer ticks
    )  +
    ggplot2::facet_grid(rows = ggplot2::vars(!!facet),
                        scales = if (free_y) "free_y" else "fixed",
      switch = "y"            ) +
    ggplot2::theme(
      strip.text.y.left = ggplot2::element_text(angle = 0, hjust = 0),
      strip.background = ggplot2::element_blank(),
      strip.placement  = "outside",
      plot.margin = ggplot2::margin(5.5, 5.5, 5.5, 35, "pt")
    )

  if (!is.null(vline_at)) {
    p <- p + ggplot2::geom_vline(xintercept = vline_at,
                                 linetype = vline_lty,
                                 linewidth = vline_size,
                                 color = vline_color)
  }

  ggplot2::ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                  plot = p, width = w, height = h)
  p
}




In [ ]:
#dapi
w <- 4
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

DAPI_thresh = 180

gghist <- plot_hist(
  data    = merged_df,
  x       = Mean_dapi,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_Mean_dapi_hist",
  fill = "#362fffff",
  w = w, h = h,
  vline_at = DAPI_thresh
)
gghist

In [ ]:
# Filter cells with high dapi levels
filter_dapi = FALSE

merged_df <- merged_df %>%
  mutate(
    DAPIpos = Mean_dapi < DAPI_thresh
  )

if (filter_dapi) {
  merged_df <- merged_df %>% 
    filter(Mean_dapi < DAPI_thresh)
}

In [ ]:
w <- 5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

GATA3_thresh = 0.4

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_GATA3_norm_intensity",
  w = 5, h = 3,
  palette = col_condition,
  hline_at = GATA3_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GATA3_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_GATA3_norm_hist",
  fill = "#6CD1D4",
  w = 5, h = 3,
  vline_at = GATA3_thresh)
gg_hist

In [ ]:
w <- 5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

NANOG_thresh = 0.5

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = NANOG_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_NANOG_norm_intensity",
  w = 5, h = 3,
  palette = col_condition,
  hline_at = NANOG_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = NANOG_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_NANOG_norm_hist",
  fill = "#6CD1D4",
  w = 5, h = 3,
  vline_at = NANOG_thresh)
gg_hist

In [ ]:
w <- 5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

GFP_thresh = 0.7
#GFP_thresh = 0.5

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GFP_norm ,
  color = condition,
  out_dir = out_dir,
  title = "C_GFP_intensity",
  w = 5, h = 3,
  palette = col_condition,
  hline_at = GFP_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GFP_norm ,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_GFP_hist",
  fill = "#6EA537",
  w = 5, h = 3,
  vline_at = GFP_thresh
)
gg_hist

In [ ]:
w <- 5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

mcherry_thresh = 80
#GFP_thresh = 0.5

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = Mean_mcherry ,
  color = condition,
  out_dir = out_dir,
  title = "C_mcherry_intensity",
  w = 5, h = 3,
  palette = col_condition,
  hline_at = mcherry_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = Mean_mcherry ,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 1,
  title   = "D_mcherry_hist",
  fill = "#CA4F33",
  w = 5, h = 3,
  vline_at = mcherry_thresh
)
gg_hist

In [ ]:
title = "GFP_mcherry_scatter"

w <- 6
h <- 4
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x =  Mean_dapi, y = Mean_mcherry, color = condition)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = "Mean_dapi", y = " mean_mcherry", title = title) +
  settheme+
   scale_color_manual(values=col_condition)+
  #geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "blue") 
  facet_wrap(~sample_name)+
  geom_vline(xintercept = GFP_thresh, linetype = "dashed", color = "grey")+
  geom_hline(yintercept = mcherry_thresh, linetype = "dashed", color = "grey")



ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

### E) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 4, h = 1.5,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 1.1,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "% positive", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggplot2::ggsave(file.path(out_dir, sprintf("%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# thresholds (edit if you want different cutoffs)
thr <- list(
  GATA3_norm = GATA3_thresh,
  GFP_norm= GFP_thresh,
  NANOG_norm= NANOG_thresh,
  mcherry_norm= mcherry_thresh
)

summary_df <- merged_df %>%
  group_by(image, sample_name, condition) %>%
  summarise(
    n = n(),
    pct_GATA3 = 100 * mean(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOG_norm > thr$NANOG_norm, na.rm = TRUE),
    pct_GFP = 100 * mean(GFP_norm> thr$GFP_norm, na.rm = TRUE),
    pct_mcherry = 100 * mean(Mean_mcherry> thr$mcherry_norm, na.rm = TRUE),
    pct_negative = 100 * mean(NANOG_norm < thr$NANOG_norm & GATA3_norm < thr$GATA3_norm),
    #pct_double_GFP_caspase3 = 100 * mean(GFP_norm > thr$GFP_norm & caspase3_norm > thr$caspase3_norm),
    .groups = "drop"
  )

head(summary_df)


In [ ]:
# find average intensities after subtyping
merged_df <- merged_df %>%
  mutate(
    GATA3pos = GATA3_norm > thr$GATA3_norm,
    NANOGpos = NANOG_norm > thr$NANOG_norm,
    mcherrypos = Mean_mcherry> thr$mcherry_norm
  )

In [ ]:
# Plot %GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3,out_dir = out_dir,
                    title = "E_pctGATA3+")
# Plot %NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG,out_dir = out_dir,
                    title = "E_pctNANOG+")

# Plot %mcherry+
plot_pct_bar_points(summary_df, pct = pct_mcherry,out_dir = out_dir,
                    title = "E_pctmcherry+")

# Plot %GFP+
plot_pct_bar_points(summary_df, pct = pct_GFP,out_dir = out_dir,
                    title = "E_pctGFP+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative,out_dir = out_dir, title = "E_pctnegative")



### plot by RFP pos or negative

In [ ]:
merged_df <- merged_df |>
  mutate(
    RFP = if_else(Mean_mcherry > thr$mcherry_norm, "pos", "neg")
  )

In [ ]:


summary_df <- merged_df %>%
  group_by(image, sample_name, condition, RFP) %>%
  summarise(
    n = n(),
    pct_GATA3 = 100 * mean(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOG_norm > thr$NANOG_norm, na.rm = TRUE),
    #pct_GFP = 100 * mean(GFP == "pos", na.rm = TRUE),
    pct_mcherry = 100 * mean(Mean_mcherry> thr$mcherry_norm, na.rm = TRUE),
    pct_negative = 100 * mean(NANOG_norm < thr$NANOG_norm & GATA3_norm < thr$GATA3_norm),
    #pct_double_GFP_caspase3 = 100 * mean(GFP_norm > thr$GFP_norm & caspase3_norm > thr$caspase3_norm),
    .groups = "drop"
  )

head(summary_df)

In [ ]:
title = "GATA3_RFPgroup"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df, aes(x = sample_name, y =pct_GATA3 , group= RFP)) +  # dots for each file
    stat_summary( aes(fill = RFP), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = RFP),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "grey30"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%GATA3+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_fill_manual(
    values = col_RFP_group,
    name = "mcherry"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "NANOG_RFPgroup"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df, aes(x = sample_name, y =pct_NANOG , group= RFP)) +  # dots for each file
    stat_summary( aes(fill = RFP), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = RFP),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "grey30"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%NANOG+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_fill_manual(
    values =col_RFP_group,
    name = "mcherry"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "doubleneg_RFPgroup"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df, aes(x = sample_name, y =pct_negative , group= RFP)) +  # dots for each file
    stat_summary( aes(fill = RFP), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = RFP),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "grey30"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%double negative+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_fill_manual(
    values = col_RFP_group,
    name = "mcherry"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
write.csv(merged_df, file.path(out_dir, sprintf("%s_analysis_summary.csv", EXP)), row.names = FALSE)